# Microenvironment Tutorial (10x)

Huetracerでの10x Visium HD向けマイクロエンバイロメント解析を、設定読み込みから保存まで一通り実行するノートブックです。

## Notebook Overview

This notebook demonstrates the end-to-end microenvironment analysis workflow for 10x Visium HD data in HueTracer.

- Load runtime configuration and initialize analysis environment.
- Prepare parameters and required reference files (NicheNet ligand-target table).
- Estimate microenvironments with VAE and project results to spatial coordinates.
- Optionally refine labels interactively and run downstream differential analyses.
- Save analysis outputs for cell-cell interaction and follow-up tutorials.

## Environment Setup and Imports

In [ ]:
# =========================
# Library Imports
# =========================

import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import json

import matplotlib.pyplot as plt
import seaborn as sns

import bin2cell as b2c

import ipywidgets as widgets
from IPython.display import display, clear_output

# Custom modules
import huetracer
from huetracer import ConfigSelector
from huetracer.widgets import create_nichenet_downloader_widget

# =========================
# Environment Configuration
# =========================

os.environ["NVCC_PREPEND_FLAGS"] = "--std=c++17"
os.environ["CCCL_IGNORE_DEPRECATED_CPP_DIALECT"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"
os.environ["TF_CPP_MAX_VLOG_LEVEL"] = "0"
warnings.filterwarnings("ignore", message=".*must be within the support of the distribution.*")

# =========================
# Device Setup
# =========================

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)
device_str = device.type
print(f"Using device: {device}")

# =========================
# Jupyter and Scanpy Settings
# =========================

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
pd.set_option('display.max_columns', None)
sc.set_figure_params(figsize=[10, 10], dpi=100)

# =========================
# Random Seed
# =========================

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# Utility Functions
# =========================

def clear_mem():
    """Clear memory by running garbage collection and CUDA cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# =========================
# VisiumHD config file selector
# =========================

selector = ConfigSelector(base_dir_default="/app/data/input")
selector.display()

# Loaded settings are available via selector.last_config
# e.g. cfg = selector.last_config

In [ ]:
# Runtime config binding (after clicking "Load")
if selector.last_config is None:
    raise RuntimeError("Please click 'Load' in the widget first.")

globals().update(selector.last_config)

## Set Parameters

解析対象領域、対象遺伝子、近傍セル数、下流解析で使うグループ設定をここで定義します。

In [ ]:
# =========================
# Parameters to be input
# =========================

# Area to be analyzed
mask_large_x1, mask_large_x2, mask_large_y1, mask_large_y2 = 250, 1750, 50, 1550

# Species
Species = "Human"
# Species = "Mouse"

# List of target gene names
if Species == "Human":
    target_genes = [
        'TNFSF11'
    ]
    prefix_mt = 'MT-'
else:
    target_genes = [
        'Col1a1'
    ]
    prefix_mt = 'mt-'

target_cell_type = annotation_dict['C2']
# target_cell_type = 'Airway epithelial cells (CAPN8+, ELF3+)'

Gene_to_analyze = "LIF"
# Gene_to_analyze = "CSF1"

# Definition of neighborhood cells
neighbor_cell_numbers = 19

# role = 'sender'
role = 'receiver'
each_display_num = 3

# Volcano plot groups
group1_environments = ['0', '2']
group2_environments = ['1', '3']

# =========================
# Lasso Selection image quality
# =========================
# Background image downsample factor for the interactive relabeling widgets.
#   1.0  = full resolution  (high quality, very heavy)
#   0.5  = half resolution  (good quality, moderate)
#   0.25 = quarter resolution (lightweight, recommended default)
#   0.1  = 1/10 resolution  (minimal, very fast)
lasso_downsample_factor = 0.25

# Output filenames
label_image_filename = "he_labels_image.pdf"
h5ad_sc_microenvironment_full_filename = SAMPLE_NAME + "_single_cell_microenvironment.h5ad"
save_spatial_plot_path = os.path.join(RESULTS_PATH, "cropped_spatial_plot.svg")
save_svg_path = os.path.join(RESULTS_PATH, "spatial_salvage_labels.svg")
h5ad_microenvironment_full_save_path = os.path.join(RESULTS_PATH, h5ad_sc_microenvironment_full_filename)


## Download and Prepare NicheNet Ligand-Target CSV

細胞間相互作用解析で利用するligand-target辞書を、必要に応じてダウンロードして利用可能な変数に登録します。

In [ ]:
# =========================
# Interactive widgets for CCI-related file preparation
# =========================

nichenet = huetracer.create_nichenet_downloader_widget(
    base_dir=BASE_DIR, runtime_namespace=globals()
)
nichenet.display()

## Microenvironment Estimation with VAE

ここから、核セグメンテーション済みAnnDataを読み込み、近傍情報を使って潜在表現を学習し、microenvironmentクラスタを推定します。

In [ ]:
# =========================
# Microenvironment estimation with variational autoencoder, 
# gene expression data of 18 cells around the center cell was used.
# =========================

h5ad_predicted_full_save_path = os.path.join(RESULTS_PATH, SAMPLE_NAME + "_nucleus_predicted.h5ad")
h5ad_save_path = os.path.join(RESULTS_PATH, SAMPLE_NAME + "_b2c.h5ad")
sp_adata_predicted = sc.read_h5ad(h5ad_predicted_full_save_path)
sp_adata_raw = sc.read_h5ad(h5ad_save_path)
lib_id = list(sp_adata_raw.uns['spatial'].keys())[0]
neighbor_cell_numbers = 19

# === Preprocessing ===
cell_mask = ((sp_adata_predicted.obs['array_row'] >= mask_x1_val) & 
             (sp_adata_predicted.obs['array_row'] <= mask_x2_val) & 
             (sp_adata_predicted.obs['array_col'] >= mask_y1_val) & 
             (sp_adata_predicted.obs['array_col'] <= mask_y2_val)
            )
sp_adata_microenvironment = sp_adata_predicted[cell_mask].copy()
del sp_adata_predicted, cell_mask; clear_mem()

# 1. Cell count by cell labels
group_counts = sp_adata_microenvironment.obs['predicted_cell_type'].value_counts()
valid_groups = group_counts[group_counts > 1].index.tolist()

# 2. Exclude cell types with only 1 cell count
mask = sp_adata_microenvironment.obs["predicted_cell_type"].isin(valid_groups)
filtered_adata = sp_adata_microenvironment[mask].copy()
filtered_adata.X = filtered_adata.layers["counts"].copy()
sc.pp.normalize_total(filtered_adata, target_sum = 1e6)
sc.pp.log1p(filtered_adata)

filtered_adata.raw = None

# 3. Select genes with DEG analysis
sc.tl.rank_genes_groups(
    filtered_adata,
    # groupby='scvi_predicted_labels',
    groupby='predicted_cell_type',
    #groupby='leiden_nucleus',
    method='wilcoxon',
    n_genes=100,
    use_raw=False
)

# names は structured array / recarray なのでそのまま配列化できる
names = filtered_adata.uns["rank_genes_groups"]["names"]  # shape: (n_genes, n_groups) か structured

# scanpyのバージョン差を吸収して2通り対応
if isinstance(names, np.ndarray) and names.dtype.names is not None:
    # structured array: 各クラスタがフィールド名
    arr = np.vstack([names[g] for g in names.dtype.names]).T  # (n_genes, n_groups)
else:
    # すでに普通の2D array の場合
    arr = np.asarray(names)

top_genes = np.unique(arr.astype(str).ravel())
top_genes = top_genes[top_genes != "nan"]  # 念のため

common_hvg = [g for g in top_genes.tolist() if g in filtered_adata.var_names]

sc.pp.highly_variable_genes(filtered_adata, n_top_genes=100, layer="counts", flavor='seurat_v3')
ref_hvg_100 = filtered_adata.var[filtered_adata.var['highly_variable']].index.tolist()
all_genes = set(common_hvg) | \
            set(ref_hvg_100)
final_gene_list = [g for g in all_genes if g in sp_adata_microenvironment.var_names]
del filtered_adata; clear_mem()

sc.pp.normalize_total(sp_adata_microenvironment, target_sum = 1e0)
sp_adata_microenvironment = sp_adata_microenvironment[:, final_gene_list].copy()
coords = sp_adata_microenvironment.obs[["array_row", "array_col"]].values
X = sp_adata_microenvironment.X.toarray() if hasattr(sp_adata_microenvironment.X, "toarray") else sp_adata_microenvironment.X  # shape: (n_cells, n_genes)
data_min = X.min()
data_max = X.max()
X = (X - data_min) / (data_max - data_min)
cell_types = sp_adata_microenvironment.obs['predicted_cell_type']
print("highly variable genes:", len(final_gene_list))

print(f"Device: {device}")
neighbor_cell_numbers = 19
# analysis
analyzer = huetracer.SpatialMicroenvironmentAnalyzer(coords, X, k_neighbors=neighbor_cell_numbers, device = device)
indices, microenv_data = analyzer.build_microenvironment_data()
del coords, X, indices, microenv_data; clear_mem()

vae_model = analyzer.train_vae(latent_dim=32, epochs=1000, batch_size=16384, lr=4e-4, dim_1 = 128, dim_2 = 128, weight_decay=1e-4)
analyzer.extract_latent_features()
del vae_model; clear_mem()

umap_embedding, clusters = analyzer.perform_umap_clustering(cell_type_data=cell_types)
# visualization
analyzer.visualize_results()
analyzer.visualize_scanpy_results()
huetracer.plot_all_clusters_highlights(analyzer)
huetracer.plot_all_cell_type_highlights(analyzer)

sp_adata_microenvironment.obs['predicted_microenvironment'] = analyzer.adata.obs['leiden'].astype(str).to_numpy()
sp_adata_microenvironment.obs['predicted_microenvironment'] = sp_adata_microenvironment.obs['predicted_microenvironment'].astype("category")
huetracer.create_hires_overlay_plot(sp_adata_microenvironment, lib_id, SAMPLE_NAME, RESULTS_PATH, file_name = "cell_type", color_key="predicted_cell_type")
huetracer.create_hires_overlay_plot(sp_adata_microenvironment, lib_id, SAMPLE_NAME, RESULTS_PATH, file_name = "microenvironment", color_key="predicted_microenvironment", TITLE = "MicroEnv")
del umap_embedding; clear_mem()


In [ ]:

sf_hires = sp_adata_microenvironment.uns["spatial"][lib_id]["scalefactors"].get(f"tissue_0.5_mpp_150_buffer_scalef", 1.0)
coords_raw = sp_adata_microenvironment.obsm["spatial_cropped_150_buffer"]
xy = (pd.DataFrame(coords_raw * sf_hires, 
                           columns=["x", "y"], 
                           index=sp_adata_microenvironment.obs_names)
              .join(sp_adata_microenvironment.obs["object_id"]) # マージ用にIDを結合
              .reset_index()
              .rename(columns={"index": "cell_id"}))
merged = xy.merge(sp_adata_microenvironment.obs, on="object_id", how="inner")
merged["predicted_microenvironment"] = clusters
merged["group"] = merged["predicted_microenvironment"]

hires_img = sp_adata_microenvironment.uns["spatial"][lib_id]["images"]["0.5_mpp_150_buffer"]
h, w = hires_img.shape[:2]
mergex = merged["x"]
mergey = merged["y"]
predicted_microenvironment_original = analyzer.adata.obs['leiden'].values.astype(str)
predicted_cell_type_original = sp_adata_microenvironment.obs["predicted_cell_type"].values.astype(str)
sp_adata_microenvironment.obs['predicted_microenvironment'] = sp_adata_microenvironment.obs['predicted_microenvironment'].astype('category')
sp_adata_microenvironment.obs['predicted_cell_type'] = sp_adata_microenvironment.obs['predicted_cell_type'].astype('category')
merged["predicted_microenvironment"] = merged["group"].astype('category')
del analyzer; clear_mem()


## Save Checkpoint (before Interactive Relabeling)

「Optional: Interactive Relabeling」に進む前にチェックポイントを保存します。  
保存後にカーネルを再起動し、下の **「Resume from Checkpoint」** セルから作業を再開できます。

In [ ]:
# =========================
# Save Checkpoint
# =========================
# VAE解析完了後、Interactive Relabeling前にチェックポイントを保存します。
# 保存後にカーネルを再起動し、下の「Load Checkpoint」セルから再開できます。

_ckpt_dir = os.path.join(RESULTS_PATH, "checkpoint_before_relabeling")
os.makedirs(_ckpt_dir, exist_ok=True)

# AnnData (sp_adata_microenvironment)
sp_adata_microenvironment.write_h5ad(os.path.join(_ckpt_dir, "sp_adata_microenvironment.h5ad"))

# DataFrame (merged)
merged.to_parquet(os.path.join(_ckpt_dir, "merged.parquet"))

# NumPy arrays
np.save(os.path.join(_ckpt_dir, "clusters.npy"), np.array(clusters, dtype=object))
np.save(os.path.join(_ckpt_dir, "predicted_microenvironment_original.npy"), predicted_microenvironment_original)
np.save(os.path.join(_ckpt_dir, "predicted_cell_type_original.npy"), predicted_cell_type_original)

# Metadata (lib_id, h, w)
with open(os.path.join(_ckpt_dir, "meta.json"), "w") as _f:
    json.dump({"lib_id": lib_id, "h": int(h), "w": int(w)}, _f)

print(f"✅ Checkpoint saved to: {_ckpt_dir}")
print("  - sp_adata_microenvironment.h5ad")
print("  - merged.parquet")
print("  - clusters.npy")
print("  - predicted_microenvironment_original.npy")
print("  - predicted_cell_type_original.npy")
print("  - meta.json")


## Optional: Interactive Relabeling

> Optional step. Use this section if you want to manually refine predicted microenvironment or cell-type labels.

> Recommended JupyterLab extensions:
- jupyter-matplotlib
- jupyter-widgets-jupyterlab-manager
- jupyterlab-plotly

How to use:
1. Select microenvironment groups to display.
2. Choose selection mode (Lasso/Rectangle).
3. Draw region(s) on the image.
4. Apply selection and update labels.
5. Use zoom as needed.
6. Re-plot to confirm updates.

### Resume from Checkpoint

カーネル再起動後にここから再開できます。  
下のセルを実行すると、チェックポイントに保存した変数が全て復元されます。  
（「Environment Setup」「Set Parameters」セクションは先に実行しておいてください）

In [ ]:
# =========================
# Load Checkpoint
# =========================
# カーネル再起動後にこのセルを実行して変数を復元します。
# 事前に「Environment Setup」「Set Parameters」セクションを実行しておいてください。

_ckpt_dir = os.path.join(RESULTS_PATH, "checkpoint_before_relabeling")

# AnnData (sp_adata_microenvironment)
sp_adata_microenvironment = sc.read_h5ad(os.path.join(_ckpt_dir, "sp_adata_microenvironment.h5ad"))

# DataFrame (merged)
merged = pd.read_parquet(os.path.join(_ckpt_dir, "merged.parquet"))

# NumPy arrays
clusters = np.load(os.path.join(_ckpt_dir, "clusters.npy"), allow_pickle=True)
predicted_microenvironment_original = np.load(
    os.path.join(_ckpt_dir, "predicted_microenvironment_original.npy"), allow_pickle=True
)
predicted_cell_type_original = np.load(
    os.path.join(_ckpt_dir, "predicted_cell_type_original.npy"), allow_pickle=True
)

# Metadata
with open(os.path.join(_ckpt_dir, "meta.json")) as _f:
    _meta = json.load(_f)
lib_id = _meta["lib_id"]
h = _meta["h"]
w = _meta["w"]

# sp_adata_raw (Downstream Analysis で必要)
h5ad_save_path = os.path.join(RESULTS_PATH, SAMPLE_NAME + "_b2c.h5ad")
if os.path.exists(h5ad_save_path):
    sp_adata_raw = sc.read_h5ad(h5ad_save_path)
    print(f"✅ sp_adata_raw loaded from: {h5ad_save_path}")
else:
    print(f"⚠️  sp_adata_raw not found at: {h5ad_save_path}")

print(f"✅ Checkpoint loaded from: {_ckpt_dir}")
print(f"   sp_adata_microenvironment: {sp_adata_microenvironment.shape}")
print(f"   merged: {merged.shape}, clusters: {len(clusters)}, lib_id: {lib_id}")


In [ ]:
# =========================
# Check microenvironment clusters before modification
# =========================

fig = huetracer.plot_spatial_plotly_fast(
    sp_adata_microenvironment,
    color_col="predicted_microenvironment",
    basis="spatial_cropped_150_buffer",
    img_key="0.5_mpp_150_buffer",
    point_size=3,
    point_opacity=0.6,
    title="Predicted microenvironment"
)

# # If you want to see the original data...
# sc.pl.spatial(
#     sp_adata_microenvironment, color='predicted_microenvironment',
#     title='Predicted microenvironment',
#     size=20,
#     alpha_img=0.2,
#     img_key="0.5_mpp_150_buffer", basis="spatial_cropped_150_buffer",
#     legend_fontsize=5,
#     groups=None,
#     spot_size=1,
#     frameon=False
# )

In [ ]:
# =========================
# Check Predicted Cell Type before modification
# =========================

fig = huetracer.plot_spatial_plotly_fast(
    sp_adata_microenvironment,
    color_col="predicted_cell_type",
    basis="spatial_cropped_150_buffer",
    img_key="0.5_mpp_150_buffer",
    point_size=3,
    point_opacity=0.6,
    title="Predicted Cell Type"
)

# # If you want to see the original data...
# sc.pl.spatial(
#     sp_adata_microenvironment, color='predicted_cell_type',
#     title='Predicted predicted_cell_type',
#     size=20,
#     alpha_img=0.2,
#     img_key="0.5_mpp_150_buffer", basis="spatial_cropped_150_buffer",
#     legend_fontsize=5,
#     groups=None,
#     spot_size=1,
#     frameon=False
# )

In [ ]:
# =========================
# Microenvironment modification
# =========================

huetracer.lasso_selection_microenvironment(
    sp_adata_microenvironment,
    merged,
    lib_id,
    clusters,
    downsample_factor=0.05
)

In [ ]:
# =========================
# Cell type modification
# =========================

huetracer.lasso_selection_cell_type(
    sp_adata_microenvironment,
    merged,
    lib_id,
    clusters,
    downsample_factor=0.25
)

In [ ]:
# Reset modification
# 
# sp_adata_microenvironment.obs['predicted_microenvironment'] = predicted_microenvironment_original
# sp_adata_microenvironment.obs['predicted_cell_type'] = predicted_cell_type_original
# merged["predicted_microenvironment"] = predicted_microenvironment_original
# merged["predicted_cell_type"] = predicted_cell_type_original

## Downstream Analysis and Visualization

以下の解析は、`sp_adata`や設定値が事前に利用可能であることを前提にします。必要に応じて`sp_adata_raw`/`sp_adata_microenvironment`を使うよう変数を合わせてください。

In [ ]:
# Gene expression and microenvironment
plt.close('all')
%matplotlib widget
huetracer.plot.create_spatial_widget(
    sp_adata_raw, 
    sp_adata_microenvironment
)

In [ ]:
# Gene expression difference among clusters (recommended workflow)
plt.close('all')
%matplotlib inline
deg_results_df = huetracer.plot.plot_deg_by_microenvironment(
    sp_adata_raw=sp_adata_raw,
    sp_adata_microenvironment=sp_adata_microenvironment,
    target_cell_type=target_cell_type,
    n_genes=8, # トップ8遺伝子を表示
    save=True,  # プロットをPDFで保存
    save_path_for_today=RESULTS_PATH
)

### Legacy DEG Cell (for comparison)

This legacy cell is kept for comparison with `huetracer.plot.plot_deg_by_microenvironment()`.

Common points:
1. Filter by target cell type and compare differential expression across predicted microenvironments.
2. Use Wilcoxon-based `rank_genes_groups`.
3. Visualize with heatmap and dotplot.

Specific points in function version (`plot_deg_by_microenvironment`):
1. Uses bin-count-aware normalization (`binned_normalized`) for DEG input.
2. Supports optional spatial masking and internal annotation alignment.
3. Uses a compact API and standardized plotting/saving flow.

Specific points in this legacy cell:
1. Performs manual preprocessing and manual result table assembly in-cell.
2. Uses `sp_adata`-centric assumptions and explicit step-by-step operations.
3. Suitable for debugging/inspection, but less reusable than the function API.

In [ ]:
# Gene expression difference among clusters (legacy workflow; kept for comparison)
# NOTE: This cell is disabled by default to avoid accidental execution.
# Set RUN_LEGACY_DEG = True only when you explicitly want to compare with function workflow.
RUN_LEGACY_DEG = False
if not RUN_LEGACY_DEG:
    raise RuntimeError(
        "Legacy comparison cell is disabled by default. "
        "Set RUN_LEGACY_DEG=True to run this cell intentionally."
    )

# NOTE: This cell assumes `sp_adata` is prepared and contains
# `predicted_cell_type` and `predicted_microenvironment` in `.obs`.

target_cell_type = "Epithelial_Tumor_CEACAM"
print(f"Unique cell types in sp_adata: {sp_adata.obs['predicted_cell_type'].unique()}")
if target_cell_type not in sp_adata.obs['predicted_cell_type'].unique():
    print(f"Error: The specified target_cell_type '{target_cell_type}' is not found in sp_adata.obs['predicted_cell_type']. Please check the spelling or available cell types.")
sp_adata_filtered_by_celltype = sp_adata[sp_adata.obs['predicted_cell_type'] == target_cell_type].copy()

if sp_adata_filtered_by_celltype.n_obs == 0:
    print(f"Error: After filtering by '{target_cell_type}', no cells remain. This might be due to an incorrect cell type name or very sparse data.")

print(f"Analyzing {sp_adata_filtered_by_celltype.n_obs} cells for '{target_cell_type}'.")

sc.pp.normalize_total(sp_adata_filtered_by_celltype, target_sum=1e4)
sc.pp.log1p(sp_adata_filtered_by_celltype)

group_counts = sp_adata_filtered_by_celltype.obs['predicted_microenvironment'].value_counts()
# only microenvironment including >9 cells
valid_groups = group_counts[group_counts > 9].index.tolist()

if len(valid_groups) < 2:
    print("Error: Less than 2 valid groups with more than 1 cell.")
    print(f"Valid groups: {valid_groups}")
    exit()

sp_adata_filtered_by_celltype = sp_adata_filtered_by_celltype[sp_adata_filtered_by_celltype.obs['predicted_microenvironment'].isin(valid_groups)].copy()

# rank_genes_groups
key_added_for_rank_genes = f'rank_genes_groups_by_microenvironment_in_{target_cell_type}'
sc.tl.rank_genes_groups(
    sp_adata_filtered_by_celltype,
    groupby='predicted_microenvironment',
    method='wilcoxon',
    use_raw=False,
    key_added=key_added_for_rank_genes
)

# ----------------------------------------------------
# Visualization
# ----------------------------------------------------
print(f"\nDifferentially expressed genes for {target_cell_type} in each microenvironment:")
group_counts = sp_adata_filtered_by_celltype.obs['predicted_microenvironment'].value_counts()
valid_groups = [g for g in group_counts.index if group_counts[g] > 9]
result = sp_adata_filtered_by_celltype.uns[key_added_for_rank_genes]
groups = result['names'].dtype.names

all_gene_data = []
for group in groups:
    if group not in valid_groups:
        continue
    top_genes = result['names'][group][:10].tolist()
    top_scores = result['scores'][group][:10].tolist()
    top_pvals_adj = result['pvals_adj'][group][:10].tolist()
    for i in range(len(top_genes)):
        all_gene_data.append({
            'microenvironment': group,
            'gene_name': top_genes[i],
            'score': top_scores[i],
            'pvals_adj': top_pvals_adj[i]
        })

gene_rank_df = pd.DataFrame(all_gene_data)
print(gene_rank_df)

print(f"\nGenerating heatmap for {target_cell_type} by predicted_microenvironment...")
sc.pl.rank_genes_groups_heatmap(
    sp_adata_filtered_by_celltype,
    groupby='predicted_microenvironment',
    key=key_added_for_rank_genes,
    n_genes=10,
    min_logfoldchange=0.5,
    show_gene_labels=True,
    use_raw=False,
    cmap='viridis',
    save=f'_{target_cell_type}_microenvironment_genes_heatmap.pdf'
)
plt.show()

print(f"\nGenerating dotplot for {target_cell_type} by predicted_microenvironment...")
sc.pl.rank_genes_groups_dotplot(
    sp_adata_filtered_by_celltype,
    groupby='predicted_microenvironment',
    key=key_added_for_rank_genes,
    n_genes=5,
    min_logfoldchange=0.5,
    standard_scale='var',
    save=f'_{target_cell_type}_microenvironment_genes_dotplot.pdf'
)
plt.show()

print("\nLegacy analysis complete.")

In [ ]:
# ---------- 5. 色マッピング ----------
group_order = sorted(merged["predicted_microenvironment"].dropna().unique())
# マーカー形状リスト（Seabornで使えるmatplotlibのmarker）
markers = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'X', 'P', 'H', '8', 'd', '|']
# カラーパレットを増強（例："tab20"）
palette = sns.color_palette("tab20", n_colors=len(group_order))

# 色とマーカーの辞書作成
color_map = dict(zip(group_order, palette))
marker_cycle = cycle(markers)
marker_map = {group: next(marker_cycle) for group in group_order}

# 描画
hires_img = sp_adata_microenvironment.uns["spatial"][lib_id]["images"]["hires"]
h, w = hires_img.shape[:2]

# 図の準備
fig = plt.figure(figsize=(6, 6), dpi=300)
ax = fig.add_axes([0, 0, 1, 1])

# 表示範囲の設定：右上1/4を切り出す
x_start, x_end = w // 2, w       # 横方向：右半分
y_start, y_end = h // 2, h       # 縦方向：上半分（反転しない）

# 表示
ax.imshow(hires_img, extent=[0, w, 0, h])  # Y軸そのまま（反転しない）
ax.set_xlim(x_start, x_end)
ax.set_ylim(y_start, y_end)
ax.axis('off')  # 軸目盛りを消す

scaling = sp_adata_microenvironment.uns["spatial"][lib_id]["scalefactors"]["tissue_hires_scalef"]
merged["x"] = sp_adata_microenvironment.obsm["spatial"][:, 0] * scaling / 2 + x_start
merged["y"] = - sp_adata_microenvironment.obsm["spatial"][:, 1] * scaling / 2 + 2 * y_start

# クラスタごとに個別に描画（scatterplotではなくplotを使う）
for group in group_order:
    data_sub = merged[merged["predicted_microenvironment"] == group]
    ax.scatter(
        data_sub["x"],
        data_sub["y"],
        c=[color_map[group]],
        marker=marker_map[group],
        s=0.5,           # 点のサイズ（調整可能）
        alpha=0.5,
        label=group,
        linewidths=0,
        rasterized=True
    )

# ax.invert_yaxis()
# ax.set_axis_off()

# 凡例を調整
ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    title="Microenvironment clustering",
    markerscale=8,
    frameon=False,
    fontsize=6
)

# ---------- 7. 保存 ----------
filename = SAMPLE_NAME + "_overlay_hires_by_microenvironment.pdf"
out_pdf = os.path.join(save_path_for_today, filename)
fig.savefig(out_pdf, format="pdf", dpi=1000, bbox_inches="tight")
plt.close(fig)

# データフレーム
df = sp_adata_microenvironment.obs

# クロス集計して割合を計算
cross_tab = pd.crosstab(df['predicted_microenvironment'], df['predicted_cell_type'])
proportions = cross_tab.div(cross_tab.sum(axis=1), axis=0)  # 行ごとに正規化

# 描画
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(proportions.values, aspect='auto', cmap='viridis')

# 軸ラベル
ax.set_xticks(np.arange(proportions.shape[1]))
ax.set_xticklabels(proportions.columns, rotation=45, ha='right')
ax.set_yticks(np.arange(proportions.shape[0]))
ax.set_yticklabels(proportions.index)

# カラーバー
cbar = ax.figure.colorbar(im, ax=ax)
cbar.set_label('Fraction of cell type')

# セル中央に％表示
for i in range(proportions.shape[0]):
    for j in range(proportions.shape[1]):
        text = f"{proportions.values[i, j]*100:.1f}%"
        ax.text(j, i, text, ha='center', va='center', color='white' if proportions.values[i,j] < 0.5 else 'black', fontsize=8)

ax.set_xlabel('Predicted Cell Type')
ax.set_ylabel('Predicted Microenvironment')
ax.set_title('Fraction of Predicted Cell Type per Microenvironment')

# 保存
filename = SAMPLE_NAME + "_celltype_and_microenvironment.pdf"
out_pdf = os.path.join(save_path_for_today, filename)
fig.savefig(out_pdf, format="pdf", dpi=1000, bbox_inches="tight")
plt.close(fig)

In [ ]:
# Volcano plot of gene expression between clusters

# group1_environments = ['1', '2', '6', '8', '9'] # Microenvironmentのカテゴリ名。文字列で定義
# group2_environments = ['0', '3', '4', '5', '7'] # Microenvironmentのカテゴリ名。文字列で定義
group1_environments = ['0', '1', '3', '4'] # Microenvironmentのカテゴリ名。文字列で定義
group2_environments = ['2', '5', '6','7', '8', '9', '10', '11', '12'] # Microenvironmentのカテゴリ名。文字列で定義

# ----------------------------------------------------
# 2. 対象細胞種のフィルタリング
# ----------------------------------------------------
# まず、sp_adata.obs['predicted_cell_type'] のユニークな値を確認
print(f"Unique cell types in sp_adata: {sp_adata.obs['predicted_cell_type'].unique()}")
if target_cell_type not in sp_adata.obs['predicted_cell_type'].unique():
    print(f"Error: The specified target_cell_type '{target_cell_type}' is not found in sp_adata.obs['predicted_cell_type']. Please check the spelling or available cell types.")
    raise SystemExit("Exiting due to missing cell type.")

# 指定した細胞種に属する細胞のみをフィルタリング
adata_for_volcano = sp_adata[
    sp_adata.obs['predicted_cell_type'] == target_cell_type
].copy()

# フィルタリング後の細胞数を確認
if adata_for_volcano.n_obs == 0:
    print(f"Error: After filtering by '{target_cell_type}', no cells remain. This might be due to an incorrect cell type name or very sparse data.")
    raise SystemExit("Exiting due to no remaining cells after filtering.")

print(f"Analyzing {adata_for_volcano.n_obs} cells for '{target_cell_type}'.")

# ----------------------------------------------------
# 3. 新しいグループ列の作成
# ----------------------------------------------------
if 'predicted_microenvironment' not in adata_for_volcano.obs.columns:
    print(f"Error: 'predicted_microenvironment' column not found in adata_for_volcano.obs. Please ensure this column exists.")
    raise SystemExit("Exiting due to missing 'predicted_microenvironment' column.")

adata_for_volcano.obs['volcano_group'] = np.nan

adata_for_volcano.obs.loc[adata_for_volcano.obs['predicted_microenvironment'].isin(group1_environments), 'volcano_group'] = 'group1'
adata_for_volcano.obs.loc[adata_for_volcano.obs['predicted_microenvironment'].isin(group2_environments), 'volcano_group'] = 'group2'

adata_for_volcano = adata_for_volcano[adata_for_volcano.obs['volcano_group'].notna()].copy()

unique_volcano_groups = adata_for_volcano.obs['volcano_group'].dropna().unique()
print(f"Unique volcano groups after categorization: {unique_volcano_groups}")

if len(unique_volcano_groups) < 2:
    print(f"Error: Less than 2 unique 'volcano_group' categories found after creating groups. Cannot perform DGE analysis.")
    raise SystemExit("Exiting due to insufficient volcano groups for DGE analysis.")

# ----------------------------------------------------
# 4. 遺伝子のフィルタリングをここに追加
# ----------------------------------------------------
print(f"Initial number of genes: {adata_for_volcano.n_vars}")

# 遺伝子発現が0の細胞が少ない、または発現細胞数が閾値以上の遺伝子のみを残す
# min_cells: その遺伝子を発現している細胞が最低限必要な数 (例: 10細胞以上)
# min_counts: その遺伝子の合計カウントが最低限必要な数 (通常は min_cells で十分)
# 遺伝子レベルでのフィルタリングなので、各細胞の合計リード数ではなく、各遺伝子をカウントします。
# 遺伝子発現が0の細胞は `min_cells` で効果的に除外されます。

# まず、生のデータ（`adata.raw`）があればそれを利用してフィルタリングの基準を計算するのが望ましいです。
# そうでなければ、現在の `adata.X` を使います。
if adata_for_volcano.raw is not None:
    # `raw` データを使って発現細胞数を計算
    # `X > 0` で非ゼロ発現の細胞数を数えます
    sc.pp.filter_genes(adata_for_volcano, min_cells=10, groupby='volcano_group')
else:
    # `raw` データがなければ、現在の `adata.X` を使用
    # `min_cells=10` で10細胞以上で発現している遺伝子を残す
    sc.pp.filter_genes(adata_for_volcano, min_cells=100)
print(f"Initial number of genes: {adata_for_volcano.n_vars}")

print(f"Number of genes after filtering (min_cells=10 per group): {adata_for_volcano.n_vars}")

# ----------------------------------------------------
# 5. 正規化と対数変換 (DGEの前処理)
# ----------------------------------------------------
sc.pp.normalize_total(adata_for_volcano, target_sum=1e4)
sc.pp.log1p(adata_for_volcano)

# ----------------------------------------------------
# 6. 差次的発現解析 (DGE) の実行
# ----------------------------------------------------
key_added_dge = f'dge_group1_vs_group2_in_{target_cell_type}'
sc.tl.rank_genes_groups(
    adata_for_volcano,
    groupby='volcano_group',
    reference='group2',
    method='wilcoxon',
    use_raw=False,
    key_added=key_added_dge
)

if key_added_dge not in adata_for_volcano.uns:
    print(f"Error: Differential gene expression results with key '{key_added_dge}' not found in .uns after execution. Check for previous errors/warnings from sc.tl.rank_genes_groups.")
    raise SystemExit("Exiting due to missing DGE results.")

# ----------------------------------------------------
# 7. Volcano Plot の描画
# ----------------------------------------------------
result = adata_for_volcano.uns[key_added_dge]

dge_results_df = pd.DataFrame({
    'gene': result['names']['group1'],
    'log2fc': result['logfoldchanges']['group1'],
    'pvals_adj': result['pvals_adj']['group1']
})

dge_results_df['pvals_adj'].replace(0, np.finfo(float).eps, inplace=True)
dge_results_df['-log10_pvals_adj'] = -np.log10(dge_results_df['pvals_adj'])

log2fc_threshold = 0.5
pval_threshold = 0.05
neg_log10_pval_threshold = -np.log10(pval_threshold)

dge_results_df['significant'] = (
    (dge_results_df['pvals_adj'] < pval_threshold) &
    (np.abs(dge_results_df['log2fc']) > log2fc_threshold)
)

# 新しい列 'abs_log2fc' を追加
dge_results_df['abs_log2fc'] = np.abs(dge_results_df['log2fc'])

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=dge_results_df,
    x='log2fc',
    y='-log10_pvals_adj',
    hue='significant',
    palette={True: 'red', False: 'grey'},
    s=20,
    alpha=0.7,
    ax=plt.gca()
)

plt.axvline(log2fc_threshold, color='blue', linestyle='--', linewidth=1)
plt.axvline(-log2fc_threshold, color='blue', linestyle='--', linewidth=1)
plt.axhline(neg_log10_pval_threshold, color='green', linestyle='--', linewidth=1)

plt.title(f'Volcano Plot for {target_cell_type}: Group ({", ".join(group1_environments)}) vs Group ({", ".join(group2_environments)})')
plt.xlabel('Log2 Fold Change')
plt.ylabel('-Log10 (Adjusted p-value)')
plt.grid(True, linestyle=':', alpha=0.6)

top_n_genes_to_label = 20
# 'by' 引数に新しい列名 'abs_log2fc' を文字列として渡す
significant_genes_to_label = dge_results_df[dge_results_df['significant']].sort_values(
    by='abs_log2fc', ascending=False
).head(top_n_genes_to_label)

texts = []
for idx, row in significant_genes_to_label.iterrows():
    texts.append(plt.text(row['log2fc'], row['-log10_pvals_adj'], row['gene'], fontsize=9))

if texts:
    at.adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', lw=0.5))

plt.show()

# ----------------------------------------------------
# 8. 有意に変動している遺伝子の抽出 
# ----------------------------------------------------
significant_up_genes = dge_results_df[
    (dge_results_df['log2fc'] > log2fc_threshold) &
    (dge_results_df['pvals_adj'] < pval_threshold)
].sort_values(by='log2fc', ascending=False)

significant_down_genes = dge_results_df[
    (dge_results_df['log2fc'] < -log2fc_threshold) &
    (dge_results_df['pvals_adj'] < pval_threshold)
].sort_values(by='log2fc', ascending=True)

print(f"\n--- Significantly Upregulated Genes in Group ({', '.join(group1_environments)}) ---")
if not significant_up_genes.empty:
    print(significant_up_genes.head(20))
else:
    print("No significantly upregulated genes found.")

print(f"\n--- Significantly Downregulated Genes in Group ({', '.join(group1_environments)}) ---")
if not significant_down_genes.empty:
    print(significant_down_genes.head(20))
else:
    print("No significantly downregulated genes found.")

print("\nVolcano Plot analysis complete.")


In [ ]:
# Cleanup after volcano plot analysis
del adata_for_volcano, dge_results_df, significant_up_genes, significant_down_genes
clear_mem()


In [ ]:
# --- 準備 ---
n_cols = 4
n_genes = len(target_genes)
n_rows = int(np.ceil(n_genes / n_cols))

# プロット設定
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3), constrained_layout=True)
axes = axes.flatten()

for i, gene in enumerate(target_genes):
    ax = axes[i]

    # 遺伝子が存在しない場合はスキップ
    if gene not in sp_adata.var_names:
        ax.set_title(f"{gene} not found")
        ax.axis("off")
        continue

    # 発現量の抽出（スパース対応）
    idx = sp_adata.var_names.get_loc(gene)
    if hasattr(sp_adata.X, "tocsc"):
        expr = sp_adata.X[:, idx].toarray().flatten()
    else:
        expr = sp_adata.X[:, idx].flatten()

    # 統計量
    mean_val = np.mean(expr)
    std_val = np.std(expr)

    # ヒストグラム描画
    ax.hist(expr, bins=100, log=True, color="steelblue", edgecolor="black")
    ax.set_title(f"{gene}\nMean={mean_val:.2f}, SD={std_val:.2f}")
    ax.set_xlabel("Expr Level")
    ax.set_ylabel("Cell Count (log)")
    ax.grid(True, which="both", linestyle="--", alpha=0.4)

# 余ったサブプロットを消す
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.show()

n_cols = 4
n_genes = len(target_genes)
n_rows = int(np.ceil(n_genes / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3), constrained_layout=True)
axes = axes.flatten()

for i, gene in enumerate(target_genes):
    ax = axes[i]

    if gene not in sp_adata.var_names:
        ax.set_title(f"{gene} not found")
        ax.axis("off")
        continue

    # 発現量取得（スパース対応）+ 非ゼロのみ抽出
    idx = sp_adata.var_names.get_loc(gene)
    if hasattr(sp_adata.X, "tocsc"):
        expr = sp_adata.X[:, idx].toarray().flatten()
    else:
        expr = sp_adata.X[:, idx].flatten()
    expr = expr[expr > 0]

    if len(expr) == 0:
        ax.set_title(f"{gene}\nNo nonzero values")
        ax.axis("off")
        continue

    # 統計量
    mean_val = np.mean(expr)
    std_val = np.std(expr)

    # ヒストグラム描画
    ax.hist(expr, bins=100, log=True, color="steelblue", edgecolor="black")
    ax.set_title(f"{gene}\nMean={mean_val:.2f}, SD={std_val:.2f}")
    ax.set_xlabel("Expr Level (>0)")
    ax.set_ylabel("Cell Count (log)")
    ax.grid(True, which="both", linestyle="--", alpha=0.4)

# 余ったサブプロットを非表示に
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.show()

In [ ]:
# Cleanup after gene histogram analysis
plt.close('all')
clear_mem()


## Save Outputs

推定済みmicroenvironmentをh5adとして保存し、必要な設定値をJSONに書き出して後続解析（CCIなど）に引き継ぎます。

In [ ]:
# Save spatial data for cell-cell interaction analysis
sp_adata_microenvironment.write_h5ad(h5ad_microenvironment_full_save_path)

# Update config (merge additional runtime parameters into existing config file)
from huetracer.widgets import update_config_file

print(f"Config target: {B2C_CONFIG_SAVE_PATH}")

# Collect optional keys safely from current runtime
candidate_keys = [
    "SAMPLE_NAME",
    "BASE_DIR",
    "source_image_path",
    "SOURCE_IMAGE_PATH",
    "sc_filtered_path",
    "EXPRESSION_PATH",
    "EXPRESSION_PATH_8UM",
    "RESULTS_PATH",
    "DATE",
    "TMP_PATH",
    "B2C_CONFIG_SAVE_PATH",
    "annotation_dict",
    "file_nichenet",
    "h5ad_microenvironment_full_save_path",
    "mask_x1_val",
    "mask_x2_val",
    "mask_y1_val",
    "mask_y2_val"
]

add_config = {}
for k in candidate_keys:
    if k in globals():
        add_config[k] = globals()[k]

# Backward compatibility: also expose old mask aliases if available
if "mask_large_x1" in globals():
    add_config["mask_large_x1"] = mask_large_x1
if "mask_large_x2" in globals():
    add_config["mask_large_x2"] = mask_large_x2
if "mask_large_y1" in globals():
    add_config["mask_large_y1"] = mask_large_y1
if "mask_large_y2" in globals():
    add_config["mask_large_y2"] = mask_large_y2

update_config_file(
    B2C_CONFIG_SAVE_PATH, add_config, merge=True
)

print(f"💾 Merged {len(add_config)} config fields into {B2C_CONFIG_SAVE_PATH}")
print("Done! Go ahead.")